# 🚀 Fine-tuning Qwen2.5-1.5B with QLoRA — Google Colab

**Before running:** Go to `Runtime → Change runtime type → T4 GPU`

| Step | Description |
|------|-------------|
| 1 | Install dependencies |
| 2 | Upload your `.jsonl` data file |
| 3 | Load model + tokenizer |
| 4 | Configure QLoRA |
| 5 | Train |
| 6 | Save to Google Drive |
| 7 | Quick test |

In [6]:
#  Cell 1 — Verify GPU
import torch
assert torch.cuda.is_available(), " GPU not found! Go to Runtime → Change runtime type → T4 GPU"
print(f" GPU detected: {torch.cuda.get_device_name(0)}")
print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

 GPU detected: Tesla T4
   VRAM: 15.6 GB


In [7]:
#  Cell 2 — Install dependencies (run once, ~2 min)
!pip install -q transformers==4.47.0 datasets peft trl bitsandbytes accelerate

In [8]:
#  Cell 3 — Set your data file path
# HOW TO UPLOAD MANUALLY:
#   1. Click the  folder icon in the left sidebar
#   2. Drag & drop your .jsonl file there  (or click the upload ⬆ icon)
#   3. Wait until the file appears, then run this cell

DATA_FILE = "/content/general_chatbot_data.jsonl"   # ← change filename if different

import os
assert os.path.exists(DATA_FILE), f" File not found: {DATA_FILE}\n   Make sure you uploaded it to the /content folder"
print(f" File found: {DATA_FILE}")

 File found: /content/general_chatbot_data.jsonl


In [9]:
# ⚙️ Cell 4 — Configuration
import os

MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"
OUTPUT_DIR  = "/content/qwen-finetuned"          # temp Colab storage
DRIVE_DIR   = "/content/drive/MyDrive/qwen-finetuned"  # final save to Drive

# ── Hyperparameters ──────────────────────────────────────────
EPOCHS        = 3
BATCH_SIZE    = 4     # GPU → can go higher than CPU's 1
GRAD_ACCUM    = 2     # effective batch = 8
LEARNING_RATE = 2e-4
MAX_SEQ_LEN   = 512
LORA_RANK     = 16
SAVE_STEPS    = 100
LOG_STEPS     = 10

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(" Config ready")

 Config ready


In [10]:
#  Cell 5 — Load & split dataset
import json
from pathlib import Path
from datasets import Dataset

lines   = Path(DATA_FILE).read_text(encoding="utf-8").strip().split("\n")
samples = [json.loads(l) for l in lines if l.strip()]

split          = int(len(samples) * 0.9)
train_dataset  = Dataset.from_list(samples[:split])
eval_dataset   = Dataset.from_list(samples[split:])

print(f" Train: {len(train_dataset)} samples | Eval: {len(eval_dataset)} samples")

 Train: 1800 samples | Eval: 200 samples


In [11]:
# 🤖 Cell 6 — Load tokenizer & model (4-bit quantization for VRAM efficiency)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# 4-bit quantization config  ← key difference from CPU script
bnb_config = BitsAndBytesConfig(
    load_in_4bit               = True,
    bnb_4bit_quant_type        = "nf4",
    bnb_4bit_compute_dtype     = torch.float16,
    bnb_4bit_use_double_quant  = True,
)

print(f" Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
tokenizer.pad_token    = tokenizer.eos_token
tokenizer.padding_side = "right"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config = bnb_config,
    device_map          = "auto",        # auto-place on GPU
    trust_remote_code   = True,
)
model.config.use_cache = False

print(" Model loaded in 4-bit")
print(f"   VRAM used: {torch.cuda.memory_allocated()/1e9:.2f} GB")

 Loading Qwen/Qwen2.5-1.5B-Instruct...
 Model loaded in 4-bit
   VRAM used: 4.31 GB


In [12]:
#  Cell 7 — Configure QLoRA
from peft import LoraConfig, get_peft_model, TaskType, prepare_model_for_kbit_training

# Required step before LoRA when using 4-bit quantization
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r              = LORA_RANK,
    lora_alpha     = LORA_RANK * 2,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj"],
    lora_dropout   = 0.05,
    bias           = "none",
    task_type      = TaskType.CAUSAL_LM,
)

model = get_peft_model(model, lora_config)
trainable, total = model.get_nb_trainable_parameters()
print(f" LoRA ready — Trainable params: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

 LoRA ready — Trainable params: 4,358,144 / 1,548,072,448 (0.28%)


In [13]:
#  Cell 8 — Train!
from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir                  = OUTPUT_DIR,
    num_train_epochs            = EPOCHS,
    per_device_train_batch_size = BATCH_SIZE,
    per_device_eval_batch_size  = BATCH_SIZE,
    gradient_accumulation_steps = GRAD_ACCUM,
    learning_rate               = LEARNING_RATE,
    lr_scheduler_type           = "cosine",
    warmup_ratio                = 0.05,
    fp16                        = True,    #  GPU → enable fp16
    bf16                        = False,
    logging_steps               = LOG_STEPS,
    eval_strategy               = "steps",
    eval_steps                  = SAVE_STEPS,
    save_steps                  = SAVE_STEPS,
    save_total_limit            = 2,
    load_best_model_at_end      = True,
    report_to                   = "none",
    dataset_text_field          = "text",
    max_seq_length              = MAX_SEQ_LEN,
    packing                     = False,
    optim                       = "paged_adamw_8bit",  # saves VRAM
)

trainer = SFTTrainer(
    model         = model,
    args          = training_args,
    train_dataset = train_dataset,
    eval_dataset  = eval_dataset,
)

total_steps = len(train_dataset) * EPOCHS // (BATCH_SIZE * GRAD_ACCUM)
print(f"  Estimated steps: {total_steps}")
print("-" * 50)

trainer.train()

Converting train dataset to ChatML:   0%|          | 0/1800 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1800 [00:00<?, ? examples/s]

Converting eval dataset to ChatML:   0%|          | 0/200 [00:00<?, ? examples/s]

...

  Estimated steps: 675
--------------------------------------------------


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)


Step,Training Loss,Validation Loss
100,1.340600,0.672495
200,1.279200,0.641843
300,1.167600,0.632108
400,1.196400,0.624374
500,1.048300,0.625219
600,1.080500,0.625343


/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.12/dist-packages/torch/_dynamo/eval_frame.py:1181: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/pyt

TrainOutput(global_step=675, training_loss=1.3332630072699654, metrics={'train_runtime': 1036.2096, 'train_samples_per_second': 5.211, 'train_steps_per_second': 0.651, 'total_flos': 7554017124421632.0, 'train_loss': 1.3332630072699654})

In [14]:
#  Cell 9 — Save locally + to Google Drive
from google.colab import drive

# Save locally first
final_local = os.path.join(OUTPUT_DIR, "final")
trainer.save_model(final_local)
tokenizer.save_pretrained(final_local)
print(f" Saved locally: {final_local}")

# Mount Drive and copy
print("\n Mounting Google Drive...")
drive.mount("/content/drive")

import shutil
os.makedirs(DRIVE_DIR, exist_ok=True)
shutil.copytree(final_local, os.path.join(DRIVE_DIR, "final"), dirs_exist_ok=True)
print(f" Backed up to Google Drive: {DRIVE_DIR}/final")

 Saved locally: /content/qwen-finetuned/final

 Mounting Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
 Backed up to Google Drive: /content/drive/MyDrive/qwen-finetuned/final


In [15]:
#  Cell 10 — Quick inference test (Self-contained)
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import os

# Ensure variables are defined even if previous cells weren't run
MODEL_NAME  = "Qwen/Qwen2.5-1.5B-Instruct"
final_local = "/content/qwen-finetuned/final"

print(f" Loading tokenizer and base model: {MODEL_NAME}...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)

test_model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype = torch.float16,
    device_map  = "auto",
)

if os.path.exists(final_local):
    print(" Loading fine-tuned LoRA weights...")
    test_model = PeftModel.from_pretrained(test_model, final_local)
else:
    print(" Warning: Fine-tuned weights not found at 'final_local'. Using base model.")

test_model.eval()

def chat(user_message: str) -> str:
    messages = [
        {"role": "system", "content": "You are a helpful, friendly assistant."},
        {"role": "user",   "content": user_message},
    ]
    text   = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    with torch.no_grad():
        out = test_model.generate(
            **inputs,
            max_new_tokens     = 200,
            do_sample          = True,
            temperature        = 0.7,
            top_p              = 0.9,
            repetition_penalty = 1.1,
            pad_token_id       = tokenizer.eos_token_id,
        )
    new_tokens = out[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

test_questions = [
    "salam cv bikhir",
    "roadmap for learning ai and automation with networking using algorithme of machine learning",
    "mein name ist ammar spreche deutsch lang",
]

for q in test_questions:
    print(f"\n⡁ {q}")
    print(f"⡇ {chat(q)}")

print("\n--- Inference test complete! ---")

# --- Interactive Placeholder ---
print("\n[Interactive Mode]")
user_input = input("Type your question here (or 'exit'): ")
if user_input.lower() != 'exit':
    print(f"\n✨ Model Response: {chat(user_input)}")

 Loading tokenizer and base model: Qwen/Qwen2.5-1.5B-Instruct...


 Loading fine-tuned LoRA weights...

⡁ salam cv bikhir
⡇ Bonjour! Je peux vous aider à créer un CV professionnel ?

⡁ roadmap for learning ai and automation with networking using algorithme of machine learning
⡇ Sure! Here's a roadmap to help you learn AI, Automation, Networking, and Machine Learning (ML) effectively:

### 1. **Foundation in Computer Science**
   - **Python Basics**: Learn basic syntax, control structures, functions.
   - **Data Structures**: Understand lists, dictionaries, sets, tuples.

### 2. **Machine Learning Fundamentals**
   - **Linear Algebra**: Vectors, matrices, eigenvalues, eigenvectors.
   - **Calculus**: Partial derivatives, gradients, optimization techniques like gradient descent.
   - **Probability Theory**: Bayes' theorem, random variables, distributions.

### 3. **Deep Learning Libraries**
   - **TensorFlow/Keras**: Start coding simple neural networks using TensorFlow or Keras.
   - **PyTorch**: Experiment with PyTorch’s dynamic computation graph.

###